# 03 - Modelling

Compare four ways of handling the 0.17% fraud rate, and pick the winner by **business cost**, not accuracy.

| model | imbalance strategy |
|---|---|
| `logreg_balanced` | logistic regression, class weights |
| `logreg_smote` | logistic regression + SMOTE oversampling |
| `xgb_weighted` | XGBoost, `scale_pos_weight` |
| `xgb_smote` | XGBoost + SMOTE |

**Cost model** (`src/evaluate.py`): a missed fraud costs the transaction amount; a false alarm costs a fixed `FALSE_ALARM_COST`.

In [ ]:
import sys
from pathlib import Path

# make `src` importable whether the notebook runs from /notebooks or the repo root
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix

from src.config import TARGET, FIG_DIR, FALSE_ALARM_COST
from src.data_prep import load_splits
from src.evaluate import baseline_costs, cost_curve, best_threshold
from src.features import get_feature_columns
from src.model import compare_models, evaluate_on_test, save_model, split_xy

sns.set_style("whitegrid")
FIG_DIR.mkdir(parents=True, exist_ok=True)

train, val, test = load_splits()
feature_cols = get_feature_columns(train)
X_va, y_va, amt_va = split_xy(val, feature_cols)
print(f"false alarm cost assumption: {FALSE_ALARM_COST}")

## 1. Baselines every model must beat

In [ ]:
base_val = baseline_costs(y_va, amt_va)
print(f"Do nothing (never flag):  {base_val['no_model']:>10,.0f}")
print(f"Flag everything:          {base_val['flag_everything']:>10,.0f}")

## 2. Train and compare (selection on the validation set)

In [ ]:
results, fitted, val_scores = compare_models(train, val)
results.round(4)

**Reading the table:** `pr_auc` measures ranking quality on the rare class; `val_cost` is the total cost at each model's *own* best threshold.
The cheapest model wins - not necessarily the highest AUC, since AUC ignores how much money each error costs.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, scores in val_scores.items():
    p, r, _ = precision_recall_curve(y_va, scores)
    ax.plot(r, p, label=f"{name} (AP={average_precision_score(y_va, scores):.3f})")
ax.axhline(y_va.mean(), ls="--", color="grey", label="random guess")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Precision-Recall curves (validation)")
ax.legend()
plt.savefig(FIG_DIR / "03_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Choosing the threshold by cost (not 0.5)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name, scores in val_scores.items():
    curve = cost_curve(y_va, scores, amt_va)
    ax.plot(curve["threshold"], curve["cost"], label=name)
    best = curve.loc[curve["cost"].idxmin()]
    ax.scatter(best["threshold"], best["cost"], zorder=5)
ax.axhline(base_val["no_model"], ls="--", color="grey", label="do nothing")
ax.set_xlabel("Decision threshold"); ax.set_ylabel("Total cost on validation set")
ax.set_title("Total business cost vs decision threshold (dots = optimum)")
ax.legend()
plt.savefig(FIG_DIR / "03_cost_vs_threshold.png", dpi=150, bbox_inches="tight")
plt.show()

The curves are jagged on purpose: with only a few hundred frauds, a **single large fraud** crossing the threshold moves the total cost
noticeably. That is also why the optimum is picked on validation and then confirmed on a separate test set rather than trusted blindly.

## 4. Final evaluation on the untouched test set

In [ ]:
best_name = results.loc[0, "model"]
best_thr = float(results.loc[0, "best_threshold"])
print(f"Selected on validation: {best_name} @ threshold {best_thr:.3f}")

final = evaluate_on_test(fitted[best_name], best_thr, test, feature_cols)
X_te, y_te, amt_te = split_xy(test, feature_cols)

tn, fp, fn, tp = confusion_matrix(y_te, final["pred"]).ravel()
print(f"\nTest set: frauds caught {tp}/{tp+fn}  |  false alarms {fp:,}  |  missed frauds {fn}")
print(f"Test PR-AUC {final['pr_auc']:.3f}  |  ROC-AUC {final['roc_auc']:.3f}")

In [ ]:
costs = pd.Series({
    "Do nothing": final["baselines"]["no_model"],
    "Flag everything": final["baselines"]["flag_everything"],
    f"Model ({best_name})": final["cost"],
})
fig, ax = plt.subplots(figsize=(7, 4))
costs.plot(kind="bar", color=["grey", "grey", "seagreen"], ax=ax)
ax.set_ylabel("Total cost on test set"); ax.set_title("Business cost: model vs naive strategies")
plt.xticks(rotation=0)
for i, v in enumerate(costs):
    ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom")
plt.savefig(FIG_DIR / "03_test_cost_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"The model removes {final['savings_vs_no_model']:.1%} of the loss you would suffer by doing nothing.")

## 5. Sensitivity: what if the false-alarm cost assumption is wrong?

The $10 false-alarm cost is an assumption. A good analysis shows how the decision changes if it is off - the optimal threshold should move.

In [ ]:
scores_best = val_scores[best_name]
rows = []
for fac in [1, 5, 10, 25, 50, 100]:
    thr, cost = best_threshold(y_va, scores_best, amt_va, false_alarm_cost=fac)
    curve = cost_curve(y_va, scores_best, amt_va, false_alarm_cost=fac)
    r = curve.loc[curve["threshold"] == thr].iloc[0]
    rows.append({"false_alarm_cost": fac, "best_threshold": thr, "recall": r["recall"],
                 "precision": r["precision"], "flagged": int(r["flagged"])})
sens = pd.DataFrame(rows).round(3)
sens

As false alarms get more expensive, the optimal threshold rises and the model becomes more conservative (flags fewer transactions, accepts missing more fraud).
This is the kind of trade-off a risk team would actually negotiate - and why a fixed 0.5 threshold is the wrong default.

In [ ]:
save_model(fitted[best_name], best_name, best_thr, feature_cols)
print("Saved best model -> models/best_model.joblib")